# Week 3 Objectives

1. Baseline computation — a single number per signal per ride (avg HR and avg power from minutes 5–15)
1. Delta features — current value minus baseline (how far has HR drifted from its anchor?)
1. Rolling statistics — mean and slope over a sliding time window (is the drift accelerating?)

### Task 1 — Compute the per-ride baseline
From your clean records_filled DataFrame, extract all rows where the elapsed time since the first timestamp is between 5 and 15 minutes. Compute:

- baseline_hr — mean heart rate in that window
- baseline_power — mean power in that window

Acceptance criteria:

- Both are single float values, not a DataFrame
- You can print them and they look physiologically plausible (HR somewhere 110–150, power somewhere 100–250 for a typical ride)
- You handle the case where the baseline window contains is_pause = True rows — those should be excluded

First of all I'm going to define a function to import a a fit file. And get the records.

In [111]:
from fitparse import FitFile
import pandas as pd

activity = FitFile('22105359350_ACTIVITY.fit')

def get_records (fit_file,record_type,amount=None):
    all_records = [] ## Defines an array for all messages in the file
    for count,message in enumerate(fit_file.get_messages (record_type)): ## iterates through all messages of a certain type
        if amount is not None and count >= amount: ## sets a limit in case the user wants a limit
            break
        ## here I define what I want to see, a message is structured in fields. Therfore I want to have an dictionary with all fields of a message
        field_content = {
            field.name : field.value
            for field in message if field.name and 'unknown' not in field.name
        }
        all_records.append (field_content)
    return all_records

def get_dataframe (fit_file,record_type,index=None):
    if index:
        df = pd.DataFrame(get_records(fit_file,record_type)).set_index(index)
    else:
        df = pd.DataFrame(get_records(fit_file,record_type))
    return df

def find_gaps (timestamps):
    time_gap = []
    for count,timestamp in enumerate(timestamps):
        if count > 0:
            timestampdiff = timestamp - timestamps[count-1] 
            if timestampdiff.seconds > 1:
                time_gap.append ({
                    'start_time': timestamps[count-1],
                    'end_time': timestamp,
                    'gap': timestampdiff.seconds
                })
    return time_gap



def get_cleaned_records (fit_file):
    records = get_dataframe (fit_file,'record','timestamp')
    timestamps = records.index
    gaps = find_gaps (timestamps)
    if len(gaps) > 0:
        print ('Gaps in the file')
        print (gaps)
    else:
        print ('No Gaps Found!')
    records['cadence'] = records['cadence'].ffill(limit=10)
    records['heart_rate'] = records['heart_rate'].ffill(limit=3)
    records['power'] = records['power'].ffill(limit=1)
    return records

def get_baseline_timestamps(timestamps,baseline_minute_start,baseline_minute_end):
    for count,timestamp in enumerate(timestamps):
        timediff = timestamp - timestamps[0]
        if timediff.seconds == baseline_minute_start*60:
            baseline_start = timestamp
        if timediff.seconds == baseline_minute_end*60:
            baseline_end = timestamp 
            return [baseline_start,baseline_end]
        
def get_baselines (parameter,dataFrame,baseline_start,baseline_end):
    baseline_timestamps = get_baseline_timestamps(sorted(dataFrame.index),baseline_start,baseline_end)
    return dataFrame[parameter][baseline_timestamps[0]:baseline_timestamps[1]].mean()



In [112]:
records = get_cleaned_records(activity)

print (get_baselines('power',records,5,15))

Gaps in the file
[{'start_time': Timestamp('2026-03-08 13:24:53'), 'end_time': Timestamp('2026-03-08 13:25:34'), 'gap': 41}, {'start_time': Timestamp('2026-03-08 13:30:00'), 'end_time': Timestamp('2026-03-08 13:30:26'), 'gap': 26}, {'start_time': Timestamp('2026-03-08 13:32:34'), 'end_time': Timestamp('2026-03-08 13:33:04'), 'gap': 30}, {'start_time': Timestamp('2026-03-08 13:47:13'), 'end_time': Timestamp('2026-03-08 13:47:24'), 'gap': 11}, {'start_time': Timestamp('2026-03-08 13:53:55'), 'end_time': Timestamp('2026-03-08 13:54:02'), 'gap': 7}, {'start_time': Timestamp('2026-03-08 13:55:46'), 'end_time': Timestamp('2026-03-08 13:56:10'), 'gap': 24}, {'start_time': Timestamp('2026-03-08 14:04:19'), 'end_time': Timestamp('2026-03-08 14:04:29'), 'gap': 10}, {'start_time': Timestamp('2026-03-08 14:06:18'), 'end_time': Timestamp('2026-03-08 14:06:37'), 'gap': 19}, {'start_time': Timestamp('2026-03-08 14:06:43'), 'end_time': Timestamp('2026-03-08 14:08:19'), 'gap': 96}, {'start_time': Times